# Insightia — Notebook 02 (réécrit) : Bloc 2 — Irritants majeurs

**Objectif** : identifier et hiérarchiser les **principaux problèmes vécus par les clients**.

Ce notebook produit une table principale **motif → volume → sentiment → note → situations → preuves**.

## Sorties
- `outputs/block2_theme_summary.csv`
- `outputs/block2_theme_keywords.csv`
- `outputs/block2_theme_verbatims.csv`
- `outputs/block2_motif_situations.csv`
- `outputs/block2_summary.json`

## Question
**« Quels sont les principaux problèmes vécus par les clients ? »**


## 1. Imports & chargement (CSV sans en-tête + Parquet si dispo)

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

DATA_PATH = Path("data/commentaires_assurance_auto.csv")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
PARQUET = OUT_DIR / "comments_clean.parquet"

COLS = [
    "id","date","device","canal","sentiment","motif","contexte",
    "anciennete","formule","note","urgence","region","support","commentaire"
]

if PARQUET.exists():
    df = pd.read_parquet(PARQUET)
else:
    df = pd.read_csv(DATA_PATH, sep=";", header=None, names=COLS, encoding="utf-8")

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["note"] = pd.to_numeric(df["note"], errors="coerce")
df["urgence"] = pd.to_numeric(df["urgence"], errors="coerce")

for c in ["motif","sentiment","canal","device","contexte","region","support"]:
    if c in df.columns:
        df[c] = df[c].fillna("UNKNOWN").astype(str).str.strip()

df.head(3)


## 2. Contrôles minimum

In [ ]:

required = ["motif","sentiment","commentaire"]
missing = [c for c in required if c not in df.columns]
print("Colonnes requises manquantes:", missing)

print("Nb lignes:", len(df))
print("Nb motifs uniques:", df["motif"].nunique())
print("Nb sentiments uniques:", df["sentiment"].nunique())
print("Commentaires vides:", (df["commentaire"].astype(str).str.strip()=="").sum())

df[["motif","sentiment"]].head(5)


## 3. Volume par motif (thème)

In [ ]:

theme_volume = df["motif"].value_counts().reset_index()
theme_volume.columns = ["motif","n_comments"]
theme_volume["share"] = theme_volume["n_comments"] / theme_volume["n_comments"].sum()
display(theme_volume.head(20))


### Visualisation Top 15

In [ ]:

topn = 15
tv = theme_volume.head(topn).iloc[::-1]
plt.figure(figsize=(10,6))
plt.barh(tv["motif"], tv["n_comments"])
plt.title(f"Top {topn} motifs (volume)")
plt.tight_layout()
plt.show()


## 4. Sentiment par motif (volumes + ratios)

In [ ]:

sent_by_theme = df.groupby(["motif","sentiment"]).size().reset_index(name="n")
sent_pivot = sent_by_theme.pivot(index="motif", columns="sentiment", values="n").fillna(0).astype(int)
sent_pivot["total"] = sent_pivot.sum(axis=1)
sent_pivot = sent_pivot.sort_values("total", ascending=False)

for col in sent_pivot.columns:
    if col != "total":
        sent_pivot[col + "_share"] = sent_pivot[col] / sent_pivot["total"]

display(sent_pivot.head(20))


## 5. Note moyenne par motif

In [ ]:

note_by_theme = (
    df.groupby("motif")
      .agg(note_mean=("note","mean"), note_count=("note","count"))
      .reset_index()
)
display(note_by_theme.sort_values("note_count", ascending=False).head(20))


## 6. Préparation texte (tokenizer nettoyé identique Notebook 01)

In [ ]:

import re, unicodedata
from collections import Counter

def strip_accents(s):
    return ''.join(ch for ch in unicodedata.normalize('NFD', s) if unicodedata.category(ch) != 'Mn')

def normalize_text(s):
    s = str(s).lower().strip()
    s = strip_accents(s)
    s = s.replace("’","'").replace("'","")
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

STOPWORDS_LIGHT = {
    "le","la","les","un","une","des","de","du","au","aux","et","ou","mais","donc","or",
    "je","tu","il","elle","on","nous","vous","ils","elles","dans","sur","pour","par","avec","sans",
    "ce","cet","cette","ces","qui","que","quoi","dont"
}
STOPWORDS_STRUCT = {
    "cest","pas","jai","suis","etre","avoir","tout","rien","tres","trop","encore","toujours",
    "juste","peux","peut","fait","faire","vais","aller","merci","bonjour","soir","jour","fois",
    "quand","meme","alors","est"
}

def tokenize_clean(s):
    toks = normalize_text(s).split()
    return [t for t in toks if len(t)>=3 and t not in STOPWORDS_LIGHT and t not in STOPWORDS_STRUCT and not t.isdigit()]


## 7. Mots-clés par motif

In [ ]:

TOP_K = 12
MIN_COMMENTS_PER_THEME = 30

rows = []
for motif, grp in df.groupby("motif"):
    n = len(grp)
    if n < MIN_COMMENTS_PER_THEME:
        continue
    c = Counter()
    for s in grp["commentaire"].astype(str):
        c.update(tokenize_clean(s))
    for rank, (w, cnt) in enumerate(c.most_common(TOP_K), start=1):
        rows.append({"motif": motif, "n_comments": n, "rank": rank, "keyword": w, "count": cnt, "share_in_theme": cnt / n})

theme_keywords = pd.DataFrame(rows).sort_values(["n_comments","motif","rank"], ascending=[False, True, True])
display(theme_keywords.head(50))
theme_keywords.to_csv(OUT_DIR / "block2_theme_keywords.csv", index=False)


## 8. Détection de situations clients

In [ ]:

SITUATIONS = {
    "perte_temps": {"perds","temps","fou"},
    "essais_multiples": {"essais","plusieurs","malgre"},
    "blocage_parcours": {"bloque","tourne","rond","passe","limite","navance"},
    "recours_support": {"support","appel","via","chat"},
}

def detect_situations(text):
    toks = set(tokenize_clean(text))
    hits = []
    for name, words in SITUATIONS.items():
        if len(toks.intersection(words)) >= 2:
            hits.append(name)
    return hits

df["situations"] = df["commentaire"].astype(str).apply(detect_situations)
df[["motif","sentiment","situations","commentaire"]].head(5)


### Agrégation motif × situation

In [ ]:

rows = []
for _, r in df[["motif","situations"]].iterrows():
    for s in r["situations"]:
        rows.append({"motif": r["motif"], "situation": s})

motif_situation = (
    pd.DataFrame(rows)
      .value_counts()
      .reset_index(name="n")
      .sort_values("n", ascending=False)
)

display(motif_situation.head(30))
motif_situation.to_csv(OUT_DIR / "block2_motif_situations.csv", index=False)


## 9. Verbatims preuves par motif

In [ ]:

EXAMPLES_PER_THEME = 5
MAX_LEN = 280

kw_map = {}
for motif, sub in theme_keywords.groupby("motif"):
    kw_map[motif] = set(sub.sort_values("rank")["keyword"].tolist())

verbatim_rows = []
for motif, grp in df.groupby("motif"):
    kws = kw_map.get(motif, set())
    if not kws:
        continue

    def score_comment(txt):
        toks = set(tokenize_clean(txt))
        return len(toks.intersection(kws))

    scored = grp.copy()
    scored["kw_score"] = scored["commentaire"].astype(str).apply(score_comment)
    scored = scored.sort_values(["kw_score"], ascending=False)

    for _, r in scored.head(EXAMPLES_PER_THEME).iterrows():
        text = str(r["commentaire"]).strip().replace("\n"," ")
        if len(text) > MAX_LEN:
            text = text[:MAX_LEN] + "…"
        verbatim_rows.append({
            "motif": motif,
            "sentiment": r.get("sentiment", None),
            "note": r.get("note", None),
            "kw_score": int(r["kw_score"]),
            "situations": ",".join(r.get("situations", [])),
            "commentaire_extrait": text
        })

theme_verbatims = pd.DataFrame(verbatim_rows)
display(theme_verbatims.head(30))
theme_verbatims.to_csv(OUT_DIR / "block2_theme_verbatims.csv", index=False)


## 10. Table principale Bloc 2 (livrable client)

In [ ]:

main = theme_volume.copy()
main = main.merge(sent_pivot.reset_index(), on="motif", how="left")
main = main.merge(note_by_theme, on="motif", how="left")

top_sit = (
    motif_situation.sort_values(["motif","n"], ascending=[True, False])
                  .groupby("motif")
                  .head(3)
                  .groupby("motif")["situation"]
                  .apply(lambda x: ", ".join(x.tolist()))
                  .reset_index(name="top_situations")
)

main = main.merge(top_sit, on="motif", how="left")
main = main.sort_values("n_comments", ascending=False)

display(main.head(25))
main.to_csv(OUT_DIR / "block2_theme_summary.csv", index=False)


## 11. Exports & traçabilité JSON

In [ ]:

block2 = {
    "n_comments": int(len(df)),
    "n_motifs": int(df["motif"].nunique()),
    "min_comments_per_theme_keywords": int(MIN_COMMENTS_PER_THEME),
    "top_k_keywords": int(TOP_K),
    "examples_per_theme": int(EXAMPLES_PER_THEME),
    "situations_rules": {k: sorted(list(v)) for k, v in SITUATIONS.items()},
    "outputs": {
        "theme_summary": "outputs/block2_theme_summary.csv",
        "theme_keywords": "outputs/block2_theme_keywords.csv",
        "motif_situations": "outputs/block2_motif_situations.csv",
        "theme_verbatims": "outputs/block2_theme_verbatims.csv",
    }
}

with open(OUT_DIR / "block2_summary.json", "w", encoding="utf-8") as f:
    json.dump(block2, f, ensure_ascii=False, indent=2)

OUT_DIR / "block2_summary.json"


✅ Fin Notebook 02 (réécrit).

Check-list:
- outputs/block2_theme_summary.csv
- outputs/block2_theme_keywords.csv
- outputs/block2_motif_situations.csv
- outputs/block2_theme_verbatims.csv
- outputs/block2_summary.json
